In [0]:
%run "./02_standardization_engine"

## Run Pipeline — Bronze → Silver

This notebook is the orchestration entry point. It supports two modes:

1. **Single table** — standardize one specific source table (good for
   ad-hoc runs / backfills / testing).
2. **Metadata-driven (all active tables)** — reads the distinct set of
   active source tables straight from `METADATA_TABLE` and standardizes all
   of them in one run, sharing a single `_batch_id` across the batch. This
   is the recommended mode for a scheduled job, since adding a new table
   only requires a metadata entry — no notebook change needed.

Use the widgets below to control which mode runs.

## Widgets

In [0]:
dbutils.widgets.dropdown("run_mode", "single_table", ["single_table", "all_active_tables"])
dbutils.widgets.text("src_catalog", "cittapoc")
dbutils.widgets.text("src_schema", "bronze")
dbutils.widgets.text("src_table", "trips")

run_mode = dbutils.widgets.get("run_mode")

## Mode 1: Single Table

Standardizes exactly one Bronze source table, as identified by the widgets
above (defaults match the original pipeline: `bronze.abs.finance`).

In [0]:
if run_mode == "single_table":

    src_catalog = dbutils.widgets.get("src_catalog")
    src_schema = dbutils.widgets.get("src_schema")
    src_table = dbutils.widgets.get("src_table")

    standardize_table(
        src_catalog=src_catalog,
        src_schema=src_schema,
        src_table=src_table
    )

## Mode 2: All Active Tables (metadata-driven)

Pulls every distinct `(src_catalog, src_schema, src_table)` combination marked
`active = True` in `METADATA_TABLE` and runs the pipeline for each one. All
tables in the run share a single `_batch_id` so they can be traced back to the
same pipeline execution. Failures on one table are logged and do **not** stop
the rest of the batch; a summary is printed (and raised) at the end.

In [0]:
if run_mode == "all_active_tables":

    batch_id = str(uuid.uuid4())
    log(f"Starting batch {batch_id}")

    source_tables = (
        spark.table(METADATA_TABLE)
        .filter(F.col("active") == True)
        .select("src_catalog", "src_schema", "src_table")
        .distinct()
        .collect()
    )

    log(f"Found {len(source_tables)} active source table(s) to process")

    results = []

    for row in source_tables:
        source = f"{row.src_catalog}.{row.src_schema}.{row.src_table}"
        try:
            standardize_table(
                src_catalog=row.src_catalog,
                src_schema=row.src_schema,
                src_table=row.src_table,
                batch_id=batch_id
            )
            results.append({"source": source, "status": "SUCCESS", "error": None})
        except Exception as e:
            results.append({"source": source, "status": "FAILED", "error": str(e)})

    failures = [r for r in results if r["status"] == "FAILED"]

    log("---- Batch Summary ----")
    for r in results:
        log(f"{r['status']}: {r['source']}" + (f" ({r['error']})" if r["error"] else ""))

    if failures:
        failed_sources = "; ".join(failure["source"] for failure in failures)
        raise Exception(
            f"Batch {batch_id} completed with {len(failures)}/{len(results)} failure(s): "
            f"{failed_sources}"
        )

    log(f"Batch {batch_id} completed successfully: {len(results)}/{len(results)} table(s)")

In [0]:
%sql
SELECT * FROM cittapoc.silver.bank_transaction